# VIIRS City Panel Collection (2013-2024)

This notebook documents and validates the annual VIIRS nighttime lights panel for European cities.

What this notebook does:
- records the extraction method used in Google Earth Engine,
- loads the exported panel from `../data/interim/viirs_city_panel_2013_2024.csv`,
- runs practical quality checks,
- standardizes columns and structure,
- saves a cleaned dataset to `../data/processed/viirs_city_panel_clean.csv`.

This notebook does **not** perform local raster processing.

## Data source and extraction method

VIIRS annual nighttime lights were extracted in Google Earth Engine using:
- `NOAA/VIIRS/DNB/ANNUAL_V21` for 2013-2021,
- `NOAA/VIIRS/DNB/ANNUAL_V22` for 2022-2024,
- band: `average_masked`.

For each city, values were aggregated within a 25 km buffer around city center coordinates.

The Earth Engine extraction script is stored at:
- `../src/viirs_earth_engine.js`

The exported panel used here is:
- `../data/interim/viirs_city_panel_2013_2024.csv`

In [1]:
# Load dependencies and source file
import pandas as pd

input_path = "../data/interim/viirs_city_panel_2013_2024.csv"

df_raw = pd.read_csv(input_path)
print(f"Loaded {len(df_raw):,} rows from {input_path}")

df_raw.head()

Loaded 360 rows from ../data/interim/viirs_city_panel_2013_2024.csv


,system:index,city,count,country,country_code,lat,lon,max,mean,median,year,.geo
0,0_00000000000000000000,Madrid,10514,Spain,ES,40.4168,-3.7038,233.349594,31.447108,11.488772,2013,"{""type"":""Polygon"",""coordinates"":[[[-3.70379999..."
1,0_00000000000000000001,Barcelona,10661,Spain,ES,41.3851,2.1734,305.180725,19.692574,2.894157,2013,"{""type"":""Polygon"",""coordinates"":[[[2.1734,41.6..."
2,0_00000000000000000002,Paris,12147,France,FR,48.8566,2.3522,195.904816,31.725208,27.450194,2013,"{""type"":""Polygon"",""coordinates"":[[[2.3522,49.0..."
3,0_00000000000000000003,Lyon,11475,France,FR,45.7640,4.8357,226.131882,10.686162,3.465605,2013,"{""type"":""Polygon"",""coordinates"":[[[4.835700000..."
4,0_00000000000000000004,Berlin,13148,Germany,DE,52.5200,13.4050,110.558891,8.787160,4.242937,2013,"{""type"":""Polygon"",""coordinates"":[[[13.40499999..."


In [ ]:
# Validate structure and coverage before cleaning
print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

print("\nInfo:")
df_raw.info()

print("\nMissing values per column:")
print(df_raw.isna().sum().sort_values(ascending=False))

if "year" in df_raw.columns:
    print("\nRows per year:")
    print(df_raw["year"].value_counts(dropna=False).sort_index())
else:
    print("\nWARNING: 'year' column not found.")

if "city" in df_raw.columns:
    print("\nRows per city:")
    print(df_raw["city"].value_counts(dropna=False).sort_values(ascending=False))
else:
    print("\nWARNING: 'city' column not found.")

if {"city", "year"}.issubset(df_raw.columns):
    city_year_counts = df_raw.groupby("city")["year"].nunique().sort_values()
    print("\nUnique years per city:")
    print(city_year_counts)
else:
    print("\nWARNING: cannot check one row per city-year (missing 'city' and/or 'year').")

# Summary stats for core VIIRS indicators
summary_candidates = ["mean", "median", "max", "count"]
available_summary_cols = [c for c in summary_candidates if c in df_raw.columns]

if available_summary_cols:
    print("\nSummary statistics:")
    print(df_raw[available_summary_cols].describe().T[["count", "mean", "50%", "max"]].rename(columns={"50%": "median"}))
else:
    print("\nWARNING: none of the expected summary columns were found:", summary_candidates)

Shape: (360, 12)

Columns:
['system:index', 'city', 'count', 'country', 'country_code', 'lat', 'lon', 'max', 'mean', 'median', 'year', '.geo']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   system:index  360 non-null    object 
 1   city          360 non-null    object 
 2   count         360 non-null    int64  
 3   country       360 non-null    object 
 4   country_code  360 non-null    object 
 5   lat           360 non-null    float64
 6   lon           360 non-null    float64
 7   max           360 non-null    float64
 8   mean          360 non-null    float64
 9   median        360 non-null    float64
 10  year          360 non-null    int64  
 11  .geo          360 non-null    object 
dtypes: float64(5), int64(2), object(5)
memory usage: 33.9+ KB

Missing values per column:
system:index    0
city            0
count           

In [3]:
# Clean and standardize schema

df = df_raw.copy()

# Rename VIIRS metrics to explicit names
rename_map = {
    "mean": "mean_brightness",
    "median": "median_brightness",
    "max": "max_brightness",
    "count": "pixel_count",
}
df = df.rename(columns=rename_map)

# Drop system-generated index if present
if "system:index" in df.columns:
    df = df.drop(columns=["system:index"])

# Keep .geo only if explicitly needed for downstream spatial work.
# For this panel workflow, we drop it by default to keep the dataset tidy.
if ".geo" in df.columns:
    geo_non_null = df[".geo"].notna().sum()
    if geo_non_null == 0:
        print("Dropping '.geo' (all values missing).")
    else:
        print("Dropping '.geo' (not required for this tabular panel workflow).")
    df = df.drop(columns=[".geo"])

# Target column order
ordered_cols = [
    "city",
    "country",
    "country_code",
    "year",
    "lat",
    "lon",
    "mean_brightness",
    "median_brightness",
    "max_brightness",
    "pixel_count",
]

existing_ordered = [c for c in ordered_cols if c in df.columns]
remaining = [c for c in df.columns if c not in existing_ordered]
df = df[existing_ordered + remaining]

print("Columns after cleaning:")
print(df.columns.tolist())

df.head()

Dropping '.geo' (not required for this tabular panel workflow).
Columns after cleaning:
['city', 'country', 'country_code', 'year', 'lat', 'lon', 'mean_brightness', 'median_brightness', 'max_brightness', 'pixel_count']


,city,country,country_code,year,lat,lon,mean_brightness,median_brightness,max_brightness,pixel_count
0,Madrid,Spain,ES,2013,40.4168,-3.7038,31.447108,11.488772,233.349594,10514
1,Barcelona,Spain,ES,2013,41.3851,2.1734,19.692574,2.894157,305.180725,10661
2,Paris,France,FR,2013,48.8566,2.3522,31.725208,27.450194,195.904816,12147
3,Lyon,France,FR,2013,45.7640,4.8357,10.686162,3.465605,226.131882,11475
4,Berlin,Germany,DE,2013,52.5200,13.4050,8.787160,4.242937,110.558891,13148


In [4]:
# Basic sanity checks
issues = []

# 1) Duplicate city-year pairs
if {"city", "year"}.issubset(df.columns):
    duplicate_mask = df.duplicated(subset=["city", "year"], keep=False)
    duplicate_rows = df.loc[duplicate_mask, ["city", "year"]].sort_values(["city", "year"])
    if duplicate_rows.empty:
        print("OK: No duplicate (city, year) pairs.")
    else:
        issues.append(f"Found {duplicate_rows.shape[0]} rows with duplicate (city, year) pairs.")
        print("PROBLEM: Duplicate (city, year) pairs found.")
        display(duplicate_rows.head(20))
else:
    issues.append("Missing 'city' and/or 'year' column; duplicate check skipped.")
    print("PROBLEM: Cannot check duplicates without both 'city' and 'year'.")

# 2) Expected year coverage
expected_years = set(range(2013, 2025))
if "year" in df.columns:
    observed_years = set(pd.to_numeric(df["year"], errors="coerce").dropna().astype(int).unique())
    missing_years = sorted(expected_years - observed_years)
    extra_years = sorted(observed_years - expected_years)

    if not missing_years and not extra_years:
        print("OK: All expected years (2013-2024) are present.")
    else:
        if missing_years:
            issues.append(f"Missing expected years: {missing_years}")
            print("PROBLEM: Missing expected years:", missing_years)
        if extra_years:
            issues.append(f"Unexpected years found: {extra_years}")
            print("PROBLEM: Unexpected years found:", extra_years)
else:
    issues.append("Missing 'year' column; year coverage check skipped.")
    print("PROBLEM: Cannot check year coverage without 'year'.")

# 3) City representation across years
if {"city", "year"}.issubset(df.columns):
    years_per_city = df.groupby("city")["year"].nunique()
    expected_n_years = len(expected_years)
    incomplete_cities = years_per_city[years_per_city < expected_n_years].sort_values()

    if incomplete_cities.empty:
        print("OK: All cities are represented across all years.")
    else:
        issues.append(f"{len(incomplete_cities)} cities are missing one or more years.")
        print("PROBLEM: Some cities are missing years.")
        display(incomplete_cities.head(20))
else:
    issues.append("Missing 'city' and/or 'year' column; city coverage check skipped.")
    print("PROBLEM: Cannot check city coverage without both 'city' and 'year'.")

print("\nSanity check summary:")
if issues:
    for msg in issues:
        print("-", msg)
else:
    print("- No issues detected.")

OK: No duplicate (city, year) pairs.
OK: All expected years (2013-2024) are present.
OK: All cities are represented across all years.

Sanity check summary:
- No issues detected.


In [5]:
# Save cleaned panel for downstream analysis
output_path = "../data/processed/viirs_city_panel_clean.csv"

df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

Saved cleaned dataset to: ../data/processed/viirs_city_panel_clean.csv
Rows: 360 | Columns: 10
